## Leafmap Interactive Visualization — 2025 Los Angeles Fires

**Data:** CAL FIRE historical fire perimeters (`fire24_1.gdb`)

**What this notebook does:**
- Loads 2025 LA fire perimeters (Palisades, Eaton, and others) from the existing CAL FIRE GDB
- Builds an interactive choropleth map centered on the LA Basin colored by `GIS_ACRES` using Leafmap
- Saves the LA fires map as a standalone HTML file

**Terminal Instructions**
pip install leafmap
pip install localtileserver
pip install mapclassify


In [8]:
import geopandas as gpd
import leafmap.foliumap as leafmap
import folium
import os


In [9]:
# Filter to LA fires in 2025

gdf = gpd.read_file('../data/raw/fire24_1.gdb', layer='firep24_1')
print(gdf['YEAR_'].max())
print(gdf[gdf['YEAR_'] == 2025][['FIRE_NAME', 'YEAR_', 'GIS_ACRES']]
     .sort_values('GIS_ACRES', ascending=False))



2025.0
   FIRE_NAME   YEAR_     GIS_ACRES
0  PALISADES  2025.0  23448.882812
1      EATON  2025.0  14056.260742
2     HUGHES  2025.0  10396.798828
3    KENNETH  2025.0    998.737793
4      HURST  2025.0    831.385498
5      LIDIA  2025.0    347.704163


In [19]:
la_fires = gdf[gdf['YEAR_'] == 2025][['FIRE_NAME', 'YEAR_', 'GIS_ACRES', 'geometry']].to_crs(epsg=4326)
print(la_fires[['FIRE_NAME', 'YEAR_', 'GIS_ACRES']].sort_values('GIS_ACRES', ascending=False))
print(la_fires.crs)

   FIRE_NAME   YEAR_     GIS_ACRES
0  PALISADES  2025.0  23448.882812
1      EATON  2025.0  14056.260742
2     HUGHES  2025.0  10396.798828
3    KENNETH  2025.0    998.737793
4      HURST  2025.0    831.385498
5      LIDIA  2025.0    347.704163
EPSG:4326


In [20]:
print(la_fires.geometry.bounds)
print(la_fires.geometry.is_valid.all())


         minx       miny        maxx       maxy
0 -118.685859  34.029844 -118.500554  34.129354
1 -118.162058  34.161887 -118.013040  34.237827
2 -118.626914  34.475385 -118.540203  34.581407
3 -118.702270  34.161004 -118.669469  34.187756
4 -118.497002  34.321194 -118.467856  34.343189
5 -118.257169  34.427702 -118.235620  34.441013
True


In [22]:
# Create initial map. Google hybrid is a great choice to showcase burn scars.

m = leafmap.Map(center=[34.1, -118.2], zoom=9, google_map = 'HYBRID')
m.add_data(la_fires, column='GIS_ACRES', scheme='Quantiles', cmap='YlOrRd', legend_title='Acres Burned')
folium.GeoJson(
    la_fires,
    tooltip=folium.GeoJsonTooltip(fields=['FIRE_NAME', 'GIS_ACRES'], aliases=['Fire Name:', 'Acres Burned:'])
).add_to(m)
m


In [7]:
# Save as standalone HTML file
os.makedirs('../outputs', exist_ok=True)
m.to_html('../outputs/la_fires_2025.html')
print('Map saved.')

Map saved.
